# 7. Multi-Agent / Subgraphs

A compiled `StateGraph` can be used directly as a node in another graph — a
"subgraph". This is what a multi-agent system actually is in LangGraph: each
specialist agent is its own independently-built, independently-testable graph;
a supervisor graph routes to whichever specialist fits the request and treats
it as a single node.

Contrast with notebook 2's branching: that demo's specialist logic is a single
plain function per branch. Here, each specialist is its own fully compiled
`StateGraph` — you could run/test either one in isolation before ever wiring it
into the supervisor (this notebook does exactly that).

**Prerequisites:** Ollama running locally with `llama3.2` pulled.

### Setup

This cell makes the project's shared `tools`/`models` packages importable
regardless of where Jupyter's working directory actually is (it's usually
this notebook's own folder, not the repo root), and loads `.env` plus any
cached secrets in `.env.local` (populated by `scripts/lib/env.sh` the first
time you've run `scripts/start_app.sh` / `scripts/start_infra.sh`).

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

load_dotenv(project_root / ".env")
load_dotenv(project_root / ".env.local", override=True)  # cached secrets, if resolve_env() has run at least once
print("Project root on sys.path:", project_root)

## Build each specialist as a standalone graph first

In [ ]:
from typing import Literal, TypedDict

from langgraph.graph import END, START, StateGraph

from models.chat_models.ollama_models import SupportedModel, get_chat_model
from tools.math_tools import adder, divider, multiplier, subtractor

MATH_TOOLS = [adder, subtractor, multiplier, divider]
MATH_TOOLS_BY_NAME = {tool.__name__: tool for tool in MATH_TOOLS}


class SupervisorState(TypedDict):
    request: str
    category: str
    answer: str


llm = get_chat_model(SupportedModel.llama3_2)


def solve(state: SupervisorState) -> dict:
    ai_message = llm.bind_tools(MATH_TOOLS).invoke(state["request"])
    if not ai_message.tool_calls:
        return {"answer": ai_message.content}
    tool_call = ai_message.tool_calls[0]
    result = MATH_TOOLS_BY_NAME[tool_call["name"]](**tool_call["args"])
    return {"answer": f"{tool_call['name']}({tool_call['args']}) = {result}"}


math_graph = StateGraph(SupervisorState)
math_graph.add_node("solve", solve)
math_graph.add_edge(START, "solve")
math_graph.add_edge("solve", END)
math_specialist = math_graph.compile()

# Try it standalone, with no supervisor involved at all:
print(math_specialist.invoke({"request": "what is 9 times 8?", "category": "", "answer": ""}))

In [ ]:
def write(state: SupervisorState) -> dict:
    response = llm.invoke(f"Write a short, creative response to: {state['request']}")
    return {"answer": response.content}


writing_graph = StateGraph(SupervisorState)
writing_graph.add_node("write", write)
writing_graph.add_edge(START, "write")
writing_graph.add_edge("write", END)
writing_specialist = writing_graph.compile()

print(writing_specialist.invoke({"request": "a two-line poem about the moon", "category": "", "answer": ""}))

## Now wire both compiled graphs into a supervisor — as nodes

In [ ]:
def classify(state: SupervisorState) -> dict:
    response = llm.invoke(
        "Classify the following request as exactly one word, either "
        "'math' or 'writing' — respond with nothing else.\n\n"
        f"Request: {state['request']}"
    )
    category = "math" if "math" in response.content.lower() else "writing"
    return {"category": category}


def route(state: SupervisorState) -> Literal["math_specialist", "writing_specialist"]:
    return "math_specialist" if state["category"] == "math" else "writing_specialist"


supervisor_graph = StateGraph(SupervisorState)
supervisor_graph.add_node("classify", classify)
supervisor_graph.add_node("math_specialist", math_specialist)  # a compiled graph, used AS a node
supervisor_graph.add_node("writing_specialist", writing_specialist)
supervisor_graph.add_edge(START, "classify")
supervisor_graph.add_conditional_edges("classify", route)
supervisor_graph.add_edge("math_specialist", END)
supervisor_graph.add_edge("writing_specialist", END)
supervisor = supervisor_graph.compile()

In [ ]:
math_request = supervisor.invoke({"request": "What is 100 divided by 4?", "category": "", "answer": ""})
print("delegated_to:", "math_specialist" if math_request["category"] == "math" else "writing_specialist")
print("answer:", math_request["answer"])

In [ ]:
writing_request = supervisor.invoke({"request": "Write a haiku about autumn.", "category": "", "answer": ""})
print("delegated_to:", "math_specialist" if writing_request["category"] == "math" else "writing_specialist")
print("answer:", writing_request["answer"])

## 🧪 Playground

**1. A third specialist** — build a `research_specialist` (or reuse `tools.search_tools.web_search`), add a category, and rewire `classify`/`route`.

In [ ]:
# TODO: build a third specialist graph and wire it into the supervisor


**2. Test a specialist in isolation** — `math_specialist.invoke(...)` directly, bypassing the supervisor. This is exactly the "independently testable" property subgraphs give you.

In [ ]:
# TODO: math_specialist.invoke with a new request


**3. Force a misclassification** — try an ambiguous request and see which specialist handles it.

In [ ]:
# TODO: try an ambiguous request through the supervisor
